In [1]:
!pip install -U sentence-transformers transformers huggingface_hub
!pip install numpy==1.26.4 faiss-cpu pandas scikit-learn gradio

  Using cached huggingface_hub-1.21.0-py3-none-any.whl.metadata (14 kB)
Using cached huggingface_hub-1.21.0-py3-none-any.whl (721 kB)
  Using cached huggingface_hub-1.21.0-py3-none-any.whl.metadata (14 kB)
Using cached huggingface_hub-1.21.0-py3-none-any.whl (721 kB)


In [2]:
import numpy as np
print(np.__version__)

1.26.4


In [3]:
from sentence_transformers import SentenceTransformer
import pandas as pd

print("✅ Everything is working!")

✅ Everything is working!


In [5]:
!pip -q install -U sentence-transformers faiss-cpu pandas numpy scikit-learn gradio

In [6]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os

ZIP_PATH = "/content/drive/MyDrive/Bangladesh Legal Acts Dataset.zip"

if os.path.exists(ZIP_PATH):
    print("✅ Dataset found:")
    print(ZIP_PATH)
else:
    print("❌ Dataset not found. Check the filename or location.")

✅ Dataset found:
/content/drive/MyDrive/Bangladesh Legal Acts Dataset.zip


In [8]:
import zipfile
import os

EXTRACT_DIR = "/content/bangladesh_legal_dataset"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("✅ Extraction completed.")
print("Extracted to:", EXTRACT_DIR)

✅ Extraction completed.
Extracted to: /content/bangladesh_legal_dataset


In [9]:
for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        print(os.path.join(root, file))

/content/bangladesh_legal_dataset/govt.json
/content/bangladesh_legal_dataset/filtered_act_list.csv
/content/bangladesh_legal_dataset/Contextualized_Bangladesh_Legal_Acts.json
/content/bangladesh_legal_dataset/balanced_1000.csv
/content/bangladesh_legal_dataset/bangladesh_legal_systems.json
/content/bangladesh_legal_dataset/acts/act-print-831.json
/content/bangladesh_legal_dataset/acts/act-print-564.json
/content/bangladesh_legal_dataset/acts/act-print-355.json
/content/bangladesh_legal_dataset/acts/act-print-205.json
/content/bangladesh_legal_dataset/acts/act-print-1436.json
/content/bangladesh_legal_dataset/acts/act-print-306.json
/content/bangladesh_legal_dataset/acts/act-print-1235.json
/content/bangladesh_legal_dataset/acts/act-print-1494.json
/content/bangladesh_legal_dataset/acts/act-print-423.json
/content/bangladesh_legal_dataset/acts/act-print-747.json
/content/bangladesh_legal_dataset/acts/act-print-695.json
/content/bangladesh_legal_dataset/acts/act-print-123.json
/content/

In [10]:
import glob
import pandas as pd

csv_files = glob.glob(os.path.join(EXTRACT_DIR, "**", "*.csv"), recursive=True)

print("CSV files found:")
for f in csv_files:
    print("-", f)

dfs = []

for f in csv_files:
    try:
        df = pd.read_csv(f)
        df["source_file"] = os.path.basename(f)
        dfs.append(df)
        print(f"Loaded: {f} | Rows: {len(df)}")
    except Exception as e:
        print(f"Skipped {f}: {e}")

CSV files found:
- /content/bangladesh_legal_dataset/filtered_act_list.csv
- /content/bangladesh_legal_dataset/balanced_1000.csv
Loaded: /content/bangladesh_legal_dataset/filtered_act_list.csv | Rows: 1489
Loaded: /content/bangladesh_legal_dataset/balanced_1000.csv | Rows: 3160


In [11]:
combined_df = pd.concat(dfs, ignore_index=True)

print("Total rows:", len(combined_df))
print("Columns:")
print(combined_df.columns.tolist())

combined_df.head()

Total rows: 4649
Columns:
['Act Title', 'Act No', 'Act Year', 'Act Link', 'full_text', 'is_repealed', 'source_file', 'act_title', 'year', 'section', 'govt_system']


,Act Title,Act No,Act Year,Act Link,full_text,is_repealed,source_file,act_title,year,section,govt_system
0,"THE [***] WILLS AND INTESTACY REGULATION, 1799",V,1799,http://bdlaws.minlaw.gov.bd/act-1315.html,http://bdlaws.minlaw.gov.bd/act-print-1315.html,False,filtered_act_list.csv,NaN,NaN,NaN,NaN
1,THE [***] CHARITABLE ENDOWMENTS PUBLIC BUILDI...,XIX,1810,http://bdlaws.minlaw.gov.bd/act-1316.html,http://bdlaws.minlaw.gov.bd/act-print-1316.html,False,filtered_act_list.csv,NaN,NaN,NaN,NaN
2,"THE [***] STATE PRISONERS REGULATION, 1818",III,1818,http://bdlaws.minlaw.gov.bd/act-1317.html,http://bdlaws.minlaw.gov.bd/act-print-1317.html,False,filtered_act_list.csv,NaN,NaN,NaN,NaN
3,"THE [***] GOVERNMENT INDEMNITY REGULATION, 1822",XI,1822,http://bdlaws.minlaw.gov.bd/act-1318.html,http://bdlaws.minlaw.gov.bd/act-print-1318.html,False,filtered_act_list.csv,NaN,NaN,NaN,NaN
4,"THE [***] ALLUVION AND DILUVION REGULATION, 1825",XI,1825,http://bdlaws.minlaw.gov.bd/act-1319.html,http://bdlaws.minlaw.gov.bd/act-print-1319.html,False,filtered_act_list.csv,NaN,NaN,NaN,NaN


In [12]:
combined_df.isnull().sum()

Act Title      3160
Act No         3162
Act Year       3160
Act Link       3160
full_text      3160
is_repealed    3160
source_file       0
act_title      1489
year           1489
section        1532
govt_system    1489
dtype: int64

In [13]:
combined_df = combined_df.fillna("")

combined_df["combined_text"] = combined_df.astype(str).agg(" ".join, axis=1)

combined_df = combined_df[
    combined_df["combined_text"].str.strip().str.len() > 20
]

combined_df = combined_df.reset_index(drop=True)

print("Final documents:", len(combined_df))
combined_df[["combined_text"]].head()

Final documents: 4649


,combined_text
0,"THE [***] WILLS AND INTESTACY REGULATION, 1799..."
1,THE [***] CHARITABLE ENDOWMENTS PUBLIC BUILDI...
2,"THE [***] STATE PRISONERS REGULATION, 1818 III..."
3,"THE [***] GOVERNMENT INDEMNITY REGULATION, 182..."
4,"THE [***] ALLUVION AND DILUVION REGULATION, 18..."


In [14]:
SAVE_PATH = "/content/processed_legal_dataset.csv"

combined_df.to_csv(SAVE_PATH, index=False)

print("✅ Saved processed dataset:")
print(SAVE_PATH)

✅ Saved processed dataset:
/content/processed_legal_dataset.csv


In [15]:
processed_df = pd.read_csv("/content/processed_legal_dataset.csv")

print("Total documents:", len(processed_df))

processed_df.head()

Total documents: 4649


,Act Title,Act No,Act Year,Act Link,full_text,is_repealed,source_file,act_title,year,section,govt_system,combined_text
0,"THE [***] WILLS AND INTESTACY REGULATION, 1799",V,1799,http://bdlaws.minlaw.gov.bd/act-1315.html,http://bdlaws.minlaw.gov.bd/act-print-1315.html,False,filtered_act_list.csv,NaN,NaN,NaN,NaN,"THE [***] WILLS AND INTESTACY REGULATION, 1799..."
1,THE [***] CHARITABLE ENDOWMENTS PUBLIC BUILDI...,XIX,1810,http://bdlaws.minlaw.gov.bd/act-1316.html,http://bdlaws.minlaw.gov.bd/act-print-1316.html,False,filtered_act_list.csv,NaN,NaN,NaN,NaN,THE [***] CHARITABLE ENDOWMENTS PUBLIC BUILDI...
2,"THE [***] STATE PRISONERS REGULATION, 1818",III,1818,http://bdlaws.minlaw.gov.bd/act-1317.html,http://bdlaws.minlaw.gov.bd/act-print-1317.html,False,filtered_act_list.csv,NaN,NaN,NaN,NaN,"THE [***] STATE PRISONERS REGULATION, 1818 III..."
3,"THE [***] GOVERNMENT INDEMNITY REGULATION, 1822",XI,1822,http://bdlaws.minlaw.gov.bd/act-1318.html,http://bdlaws.minlaw.gov.bd/act-print-1318.html,False,filtered_act_list.csv,NaN,NaN,NaN,NaN,"THE [***] GOVERNMENT INDEMNITY REGULATION, 182..."
4,"THE [***] ALLUVION AND DILUVION REGULATION, 1825",XI,1825,http://bdlaws.minlaw.gov.bd/act-1319.html,http://bdlaws.minlaw.gov.bd/act-print-1319.html,False,filtered_act_list.csv,NaN,NaN,NaN,NaN,"THE [***] ALLUVION AND DILUVION REGULATION, 18..."


In [16]:
import os
# Force a reload of the modules if necessary, though usually a cell re-run after pip install works
import transformers
import sentence_transformers
from sentence_transformers import SentenceTransformer

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
MODEL_NAME = "BAAI/bge-base-en-v1.5"

try:
    model = SentenceTransformer(MODEL_NAME)
    print("✅ BGE model loaded successfully.")
except Exception as e:
    print(f"❌ Failed to load model: {e}")
    print("If the error persists, please go to Runtime -> Restart session and run all cells.")

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ BGE model loaded successfully.


In [17]:
MODEL_NAME = "BAAI/bge-base-en-v1.5"

model = SentenceTransformer(MODEL_NAME)

print("✅ BGE model loaded successfully.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ BGE model loaded successfully.


In [18]:
texts = (
    processed_df["combined_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

print("Number of texts:", len(texts))
print("Sample:")
print(texts[0][:500])

Number of texts: 4649
Sample:
THE [***] WILLS AND INTESTACY REGULATION, 1799 V 1799 http://bdlaws.minlaw.gov.bd/act-1315.html http://bdlaws.minlaw.gov.bd/act-print-1315.html False filtered_act_list.csv    


In [19]:
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/146 [00:00<?, ?it/s]

Embedding shape: (4649, 768)


In [20]:
np.save("/content/bge_embeddings.npy", embeddings)

print("✅ Embeddings saved.")

✅ Embeddings saved.


In [21]:
loaded_embeddings = np.load("/content/bge_embeddings.npy")

print("Loaded shape:", loaded_embeddings.shape)

Loaded shape: (4649, 768)


In [22]:
print("Embedding dimension:", loaded_embeddings.shape[1])

Embedding dimension: 768


In [26]:
import faiss

# 1. Initialize the FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

# 2. Add the embeddings to the index
index.add(embeddings)

# 3. Perform the search
query = "What is the punishment for theft in Bangladesh?"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

scores, indices = index.search(query_embedding, 3)

for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
    print("=" * 80)
    print(f"Rank: {rank}")
    print(f"Similarity: {score * 100:.2f}%")
    print(processed_df.iloc[idx]["combined_text"][:1200])
    print()

Rank: 1
Similarity: 74.18%
      balanced_1000.csv The Penal Code, 1860 1860.0 245. Whoever, without lawful authority, takes out of any mint, lawfully established in Bangladesh, any coining tool or instrument, shall be punished with imprisonment of either description for a term which may extend to seven years, and shall also be liable to fine. british Colonial

Rank: 2
Similarity: 72.22%
      balanced_1000.csv The Dhaka Metropolitan Police Ordinance, 1976 1976.0 88. Whoever has in his possession, or conveys in any manner, or offers for sale or pawn, anything which there is reason to believe to have been stolen or fraudulently obtained, shall, if he fails to account for such possession, be punishable with imprisonment for a term which may extend to one year, or with fine which may extend to two thousand taka, or with both. Militery Rule

Rank: 3
Similarity: 71.77%
      balanced_1000.csv The Penal Code, 1860 1860.0 240. Whoever, having any counterfeit coin, which is a counterfeit of Ba

In [27]:
SAVE_MODEL_PATH = "/content/bge_saved_model"

model.save(SAVE_MODEL_PATH)

print("✅ Model saved to:", SAVE_MODEL_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to: /content/bge_saved_model


In [29]:
import faiss
import os
from sentence_transformers import SentenceTransformer

# 1. Save the index created in the previous steps if it exists in memory
FAISS_INDEX_PATH = "/content/faiss.index"
if 'index' in globals():
    faiss.write_index(index, FAISS_INDEX_PATH)
    print(f"✅ Index saved to {FAISS_INDEX_PATH}")

# 2. Load saved model
model = SentenceTransformer("/content/bge_saved_model")

# 3. Load FAISS index
if os.path.exists(FAISS_INDEX_PATH):
    index = faiss.read_index(FAISS_INDEX_PATH)
    print("✅ Model and FAISS index loaded successfully.")
    print("Indexed documents:", index.ntotal)
else:
    print(f"❌ Error: {FAISS_INDEX_PATH} not found. Please ensure the index was saved.")

✅ Index saved to /content/faiss.index


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Model and FAISS index loaded successfully.
Indexed documents: 4649


In [30]:
def search_legal_documents(query, top_k=5):
    """
    Search the legal corpus using BGE embeddings + FAISS.
    """

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):

        similarity = float(score) * 100

        results.append({
            "rank": rank,
            "similarity_percent": round(similarity, 2),
            "document_index": int(idx),
            "text": processed_df.iloc[idx]["combined_text"]
        })

    return results

In [31]:
question = "What is the punishment for theft in Bangladesh?"

results = search_legal_documents(question, top_k=3)

for r in results:
    print("=" * 80)
    print(f"Rank: {r['rank']}")
    print(f"Similarity: {r['similarity_percent']}%")
    print()
    print(r["text"][:1500])
    print()

Rank: 1
Similarity: 74.18%

      balanced_1000.csv The Penal Code, 1860 1860.0 245. Whoever, without lawful authority, takes out of any mint, lawfully established in Bangladesh, any coining tool or instrument, shall be punished with imprisonment of either description for a term which may extend to seven years, and shall also be liable to fine. british Colonial

Rank: 2
Similarity: 72.22%

      balanced_1000.csv The Dhaka Metropolitan Police Ordinance, 1976 1976.0 88. Whoever has in his possession, or conveys in any manner, or offers for sale or pawn, anything which there is reason to believe to have been stolen or fraudulently obtained, shall, if he fails to account for such possession, be punishable with imprisonment for a term which may extend to one year, or with fine which may extend to two thousand taka, or with both. Militery Rule

Rank: 3
Similarity: 71.77%

      balanced_1000.csv The Penal Code, 1860 1860.0 240. Whoever, having any counterfeit coin, which is a counterfeit of

In [32]:
while True:

    query = input("\n🔎 Enter a legal question (or type quit): ").strip()

    if query.lower() == "quit":
        print("Session ended.")
        break

    if not query:
        print("⚠️ Please enter a valid legal question.")
        continue

    results = search_legal_documents(query, top_k=5)

    print("\nTop Results:\n")

    for r in results:

        print("-" * 80)
        print(f"Rank: {r['rank']}")
        print(f"Similarity: {r['similarity_percent']}%")
        print()
        print(r["text"][:1000])
        print()


🔎 Enter a legal question (or type quit): What is murder case punishment?

Top Results:

--------------------------------------------------------------------------------
Rank: 1
Similarity: 65.35%

      balanced_1000.csv The Penal Code, 1860 1860.0 396. If any one of five or more persons, who are conjointly committing dacoity, commits murder in so committing dacoity, every one of those persons shall be punished with death, or for life, or rigorous imprisonment for a term which may extend to ten years, and shall also be liable to fine. british Colonial

--------------------------------------------------------------------------------
Rank: 2
Similarity: 63.06%

      balanced_1000.csv The Navy Ordinance, 1961 1961.0 141. In awarding a sentence of death a court-martial shall, in its discretion, direct that the offender shall suffer death by being hanged by the neck until he be dead, or shall suffer death by being shot to death. Militery Rule

---------------------------------------------

In [33]:
sample_queries = [
    "What is the punishment for theft?",
    "Explain labour law.",
    "What is contract law?",
    "How is inheritance determined?",
    "How do I register property?"
]

for q in sample_queries:

    print("\n" + "=" * 100)
    print("QUESTION:", q)

    results = search_legal_documents(q, top_k=1)

    print("Best Similarity:", results[0]["similarity_percent"], "%")
    print()
    print(results[0]["text"][:800])


QUESTION: What is the punishment for theft?
Best Similarity: 76.53 %

      balanced_1000.csv The Penal Code, 1860 1860.0 411. Whoever dishonestly receives or retains any stolen property, knowing or having reason to believe the same to be stolen property, shall be punished with imprisonment of either description for a term which may extend to three years, or with fine, or with both. british Colonial

QUESTION: Explain labour law.
Best Similarity: 67.45 %

      balanced_1000.csv The Agricultural Labour (Minimum Wages) Ordinance, 1984 1984.0 2. In this Ordinance, unless there is anything repugnant in the subject or context,-(a) “agricultural labourer” means any person employed in agricultural crop production, but does not include-(i) a person employed by the Government;(ii) a person employed in a plantation as defined in clause (iii) of section 2 of the Payment of Wages Act, 1936 (IV of 1936);(iii) a person who works as a family labourer on monthly wages;(iv) a person employed by a com

In [34]:
!pip -q install --upgrade gradio

In [35]:
SIMILARITY_THRESHOLD = 40.0  # percentage

def legal_ai_search(query, top_k=5):
    query = query.strip()

    if len(query) == 0:
        return "⚠️ Please enter a legal question."

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, top_k)

    best_percent = float(scores[0][0]) * 100

    if best_percent < SIMILARITY_THRESHOLD:
        return (
            "❌ No sufficiently relevant legal information was found.\n\n"
            "Please try asking a Bangladesh law or legal question."
        )

    output = ""

    for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):

        similarity = float(score) * 100

        output += (
            f"# Result {rank}\n\n"
            f"**Similarity:** {similarity:.2f}%\n\n"
            f"{processed_df.iloc[idx]['combined_text'][:2000]}\n\n"
            f"{'='*70}\n\n"
        )

    return output

In [45]:
import gradio as gr

demo = gr.Interface(
    fn=legal_ai_search,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Example: What is the punishment for theft in Bangladesh?",
        label="🔎 Ask a Legal Question"
    ),
    outputs=gr.Markdown(label="📚 Search Results"),
    title="🇧🇩 Bangladesh Judiciary Legal AI (BGE + FAISS)",
    description=(
        "Enter a legal question in natural language. "
        "The system retrieves the most semantically relevant passages "
        "from the indexed legal corpus."
    ),
    flagging_mode="never"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://05142067e4a931ed44.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [47]:
sample_questions = [
    "What is the punishment for theft?",
    "Explain contract law.",
    "What are property rights?",
    "How is inheritance determined?",
    "What are labour law provisions?"
]

for q in sample_questions:
    print("=" * 80)
    print("Question:", q)
    print(legal_ai_search(q)[:800])
    print()

Question: What is the punishment for theft?
# Result 1

**Similarity:** 76.53%

      balanced_1000.csv The Penal Code, 1860 1860.0 411. Whoever dishonestly receives or retains any stolen property, knowing or having reason to believe the same to be stolen property, shall be punished with imprisonment of either description for a term which may extend to three years, or with fine, or with both. british Colonial


# Result 2

**Similarity:** 72.78%

      balanced_1000.csv The Code of Criminal Procedure, 1898 1898.0 (3) The offence of theft, or any offence which includes theft or the possession of stolen property, may be inquired into or tried by a Court within the local limits of whose jurisdiction such offence was committed or the property stolen was possessed by 

Question: Explain contract law.
# Result 1

**Similarity:** 70.56%

      balanced_1000.csv The Contract Act, 1872 1872.0 237. When an agent has, without authority, done acts or incurred obligations to third persons on behalf